In [20]:
import os
import pandas as pd

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

In [21]:
import dspy
lm = dspy.LM('openai/gpt-4.1-mini', temperature=1.0, max_tokens=16000, api_key=OPENAI_API_KEY)
dspy.configure(lm=lm)

# Dataset

In [22]:
import dspy
import pandas as pd

def init_dataset_from_df(df, split_ratio=(0.8, 0.1, 0.1), seed=42):
    assert abs(sum(split_ratio) - 1.0) < 1e-6, "split_ratio must sum to 1.0"
    assert {'text', 'polarization'}.issubset(df.columns), "DataFrame must have 'text' and 'polarization' columns"

    # Shuffle deterministically
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    total = len(df)

    # Split boundaries
    train_end = int(split_ratio[0] * total)
    val_end = train_end + int(split_ratio[1] * total)

    train_df = df.iloc[:train_end]
    val_df = df.iloc[train_end:val_end]
    test_df = df.iloc[val_end:]

    # Convert to DSPy examples
    def to_dspy_list(sub_df):
        return [
            dspy.Example({
                "statement": row["text"],
                "answer": row["polarization"],
            }).with_inputs("statement")
            for _, row in sub_df.iterrows()
        ]

    train_set = to_dspy_list(train_df)
    val_set = to_dspy_list(val_df)
    test_set = to_dspy_list(test_df)

    return train_set, val_set, test_set


# GEPA

In [ ]:
from data_utils import split_dataframe

In [23]:
filename = "../data/dev_phase/subtask1/train/eng.csv"
english_train_df = pd.read_csv(filename)
english_train_df = english_train_df.sample(n=10, random_state=42).reset_index(drop=True)

train_df, val_df, test_df = init_dataset_from_df(english_train_df)

In [7]:
len(train_df), len(val_df), len(test_df)

(8, 1, 1)

In [8]:
# Get the first example
example = train_df[0]

# Access its fields
print(example)                # Shows full Example object

Example({'statement': 'IEBC LISTED 1,031,645 new voters in the second phase of the national voter registration exercise that ended on Feb 06.', 'answer': 0}) (input_keys={'statement'})


In [9]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = example["answer"]
    statement = example["statement"]
    llm_answer = prediction.answer

    if llm_answer not in ['0', '1']:
        feedback_text = (
            f"The model must output only '1' or '0' to indicate whether the statement is polarizing. "
            f"You responded with '{prediction.answer}', which is invalid. "
            f"The correct label for this statement is '{correct_answer}'."
        )
        if statement:
            feedback_text += f"\n\nStatement:\n{statement}\n\n"
        feedback_text += (
            "Review the classification logic and ensure the response is strictly 'Yes' or 'No' "
            "without any justification or extra formatting."
        )
        return dspy.Prediction(score=0, feedback=feedback_text)

    # Compute correctness
    score = int(llm_answer == correct_answer)

    # Construct feedback
    if score == 1:
        feedback_text = f"Correct — the statement is labeled '{correct_answer}'."
    else:
        feedback_text = (
            f"Incorrect — the correct label is '{correct_answer}', but you predicted '{llm_answer}'."
        )
        if statement:
            feedback_text += f"\n\nStatement:\n{statement}\n"
        feedback_text += (
            "\nRe-examine the linguistic or emotional cues that indicate polarization. "
            "Polarizing statements often include adversarial framing, group identity references, "
            "or moral/emotional intensifiers. Non-polarizing statements tend to remain descriptive or factual."
        )

    return dspy.Prediction(score=score, feedback=feedback_text)

In [10]:
class GenerateResponse(dspy.Signature):
    statement = dspy.InputField()
    answer = dspy.OutputField()

program = dspy.ChainOfThought(GenerateResponse)

In [24]:
from dspy import GEPA

optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light",
    num_threads=1,
    track_stats=True,
    reflection_minibatch_size=3,
    reflection_lm=dspy.LM(model="gemini/gemini-2.5-flash", temperature=1.0, max_tokens=32000, api_key=GEMINI_API_KEY)
)

optimized_program = optimizer.compile(
    program,
    trainset=train_df,
    valset=val_df,
)

2025/12/20 16:24:41 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 384 metric calls of the program. This amounts to 42.67 full evals on the train+val set.
2025/12/20 16:24:41 INFO dspy.teleprompt.gepa.gepa: Using 1 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget.

GEPA Optimization:   0%|          | 0/384 [00:00<?, ?rollouts/s]2025/12/20 16:24:48 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 1 (0.0%)
2025/12/20 16:24:48 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.0

GEPA Optimization:   0%|          | 1/384 [00:07<46:16,  7.25s/rollouts]2025/12/20 16:24:48 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.0


Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:12<00:00,  4.10s/it]

2025/12/20 16:25:00 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 3 (0.0%)


2025/12/20 16:25:15 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for predict: You are an expert at classifying statements as polarizing or not.

**Task:**
Given a `statement`, determine if it is polarizing.

**Definition of a Polarizing Statement:**
A statement is considered **polarizing** if it:
*   Uses inflammatory, derogatory, or highly biased language (e.g., "dumpster fire democrats," "fake news").
*   Attacks or strongly criticizes specific political parties, groups, ideologies, or public figures in a divisive manner (e.g., accusing "MAGA folks" of "xenophobia").
*   Makes unsubstantiated accusations or labels intended to discredit or provoke strong negative reactions.
*   Is designed to provoke strong emotional, partisan, or divided reactions, often deepening existing political or social divides.
*   Expresses extreme opinions or views that lack neutrality and aim to alienate or antagonize opposing viewpoints.

A statement is **not polarizing** if it:
*   Repor

Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:13<00:00,  4.43s/it]

...

2025/12/20 16:29:05 INFO dspy.teleprompt.gepa.gepa: Iteration 12: Proposed new text for predict: Your task is to classify a given `statement` as either polarizing or not polarizing.

You must output only the `answer` field. The `answer` field must contain *only* '1' or '0'. Do not include any additional text, explanation, reasoning, or formatting beyond the single digit '1' or '0'.

- Output '1' if the statement is polarizing.
- Output '0' if the statement is not polarizing.

A statement is considered **polarizing** if it:
1.  Expresses strong opinions, accusations, or claims on contentious or politically sensitive issues (e.g., election integrity, legitimacy of leaders, social controversies).
2.  Is likely to cause significant division, disagreement, or strong emotional responses among different groups of people.
3.  Contains subjective language, takes a strong stance, or challenges widely accepted norms without presenting objective, verifiable facts.

A statement is considered **not 

Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:00<00:00, 239.69it/s]

2025/12/20 16:29:09 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 3 (0.0%)


2025/12/20 16:29:18 INFO dspy.teleprompt.gepa.gepa: Iteration 13: Proposed new text for predict: You are an expert at identifying polarizing statements. Your task is to analyze a given `statement` and determine if it is polarizing or not.

A statement is considered **polarizing (1)** if it:
*   Expresses strong partisan bias, hostility, or distrust towards a political party, group, or institution.
*   Uses emotionally charged, inflammatory, or insulting language (e.g., "dumpster fire," "fake news," "illegitimate president").
*   Dismisses entire sources of information or groups of people based on perceived political leanings without offering constructive criticism.
*   Questions the legitimacy of democratic processes, elections, or leaders with unsubstantiated claims of rigging or illegitimacy.
*   Aims to divide or incite strong emotional reactions along political or ideological lines.

A statement is considered **not polarizing (0)** if it:
*   Is factual, neutral, or provides genera

KeyboardInterrupt: 